In [7]:
import pandas as pd
import yfinance as yf
import time
import requests
import io
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Proper Path Setup ---
# Use the __file__ variable to get the directory of the currently running script
# This makes the paths portable and not hardcoded to your machine.
try:
    # Get the directory where this script is located (e.g., .../yfinancedata)
    THIS_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive environments (like Jupyter)
    THIS_SCRIPT_DIR = os.getcwd()
    logging.warning(f"Could not use __file__, falling back to current working directory: {THIS_SCRIPT_DIR}")


# Your original code implied SCRIPT_DIR was one level *above* this script's location
# (e.g., .../Stock-Market-Indices)
SCRIPT_DIR = os.path.normpath(os.path.join(THIS_SCRIPT_DIR, '..'))

# Your original code implied DATA_DIR was a *sibling* of SCRIPT_DIR
# (e.g., .../dcv3/data)
DATA_DIR = os.path.normpath(os.path.join(SCRIPT_DIR, '..', 'data'))

# --- File Definitions ---
INPUT_CSV = os.path.join(DATA_DIR, "master_indices_list.csv")
OUTPUT_DIR_INDIVIDUAL = os.path.join(DATA_DIR, "constituent_lists")
OUTPUT_FILE_MASTER = os.path.join(DATA_DIR, "master_constituents_list.csv")
SUMMARY_REPORT_FILE = os.path.join(DATA_DIR, "constituent_scraping_summary.csv")
# --- End of Path Setup ---


# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (Based on your research) ====================
# =====================================================================================
# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (v2 - EXPANDED) ==============================
# =====================================================================================
INDEX_CONFIG = {
    # --- Americas ---
    '^DJI': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^DJT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Transportation_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^DJU': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Utility_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^NDX': {'url': 'https://en.wikipedia.org/wiki/Nasdaq-100', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^RUI': {'url': 'https://en.wikipedia.org/wiki/Russell_1000_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^GSPC': {'url': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 'type': 'table', 'company_col': 'Security', 'ticker_col': 'Symbol', 'clean_fn': lambda s: s.replace('.', '-')},
    '^OEX': {'url': 'https://en.wikipedia.org/wiki/S%26P_100', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^MID': {'url': 'https://en.wikipedia.org/wiki/S%26P_400', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'},
    '^BVSP': {'url': 'https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.SA'},
    '^GSPTSE': {'url': 'https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^IPSA': {'url': 'https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.SN'},
    '^MXX': {'url': 'https://en.wikipedia.org/wiki/Indice_de_Precios_y_Cotizaciones', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^RUT': {'type': 'ishares_csv', 'name': 'Russell 2000'},
    '^MERV': {'url': 'https://en.wikipedia.org/wiki/MERVAL', 'type': 'navbox_list', 'company_col': 'Company Name', 'suffix': '.BA'},
    
    # --- Asia-Pacific ---
    '000300.SS': {'url': 'https://en.wikipedia.org/wiki/CSI_300_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '000016.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_50_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker symbol'},
    '000001.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'}, #No company list
    '^HSI': {'url': 'https://en.wikipedia.org/wiki/Hang_Seng_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker', 'clean_fn': lambda s: f"{int(s.split(':')[-1].strip()):04d}.HK"},
    '^BSESN': {'url': 'https://en.wikipedia.org/wiki/BSE_SENSEX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.BO'},
    '^NSEI': {'url': 'https://en.wikipedia.org/wiki/NIFTY_50', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^NSMIDCP': {'url': 'https://en.wikipedia.org/wiki/NIFTY_Next_50', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^N225': {'url': 'https://en.wikipedia.org/wiki/Nikkei_225', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Symbol', 'suffix': '.T'},
    '^KLSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_Bursa_Malaysia_KLCI', 'type': 'table', 'company_col': 'Constituent Name', 'ticker_col': 'Stock Code', 'suffix': '.KL'},
    '^STI': {'url': 'https://en.wikipedia.org/wiki/Straits_Times_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Stock symbol', 'suffix': '.SI'},
    '^SET.BK': {'url': 'https://en.wikipedia.org/wiki/SET50_Index_and_SET100_Index', 'type': 'table', 'company_col': 'Securities Name', 'ticker_col': 'Symbol', 'suffix': '.BK'},
    '^AXJO': {'url': 'https://en.wikipedia.org/wiki/S%26P/ASX_200', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    '^NZ50': {'url': 'https://en.wikipedia.org/wiki/S%26P/NZX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.NZ'},
    '^KS11': {'url': 'https://en.wikipedia.org/wiki/KOSPI', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^TWII': {'url': 'https://en.wikipedia.org/wiki/Taiwan_Capitalization_Weighted_Stock_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.TW'},
    '^AORD': {'url': 'https://en.wikipedia.org/wiki/All_Ordinaries', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    
    # --- Europe ---
    '^STOXX50E': {'url': 'https://en.wikipedia.org/wiki/EURO_STOXX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^STOXX': {'url': 'https://en.wikipedia.org/wiki/STOXX_Europe_600', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^ATX': {'url': 'https://en.wikipedia.org/wiki/Austrian_Traded_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.VI'},
    '^BFX': {'url': 'https://en.wikipedia.org/wiki/BEL_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.BR'},
    '^OMXC25': {'url': 'https://en.wikipedia.org/wiki/OMX_Copenhagen_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.CO'},
    '^OMXH25': {'url': 'https://en.wikipedia.org/wiki/OMX_Helsinki_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.HE'},
    '^FCHI': {'url': 'https://en.wikipedia.org/wiki/CAC_40', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.PA'},
    '^CN20': {'url': 'https://en.wikipedia.org/wiki/CAC_Next_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.PA'},
    '^GDAXI': {'url': 'https://en.wikipedia.org/wiki/DAX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.DE'},
    '^MDAXI': {'url': 'https://en.wikipedia.org/wiki/MDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    '^TECDAX': {'url': 'https://en.wikipedia.org/wiki/TecDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    'FTSEMIB.MI': {'url': 'https://en.wikipedia.org/wiki/FTSE_MIB', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^AEX': {'url': 'https://en.wikipedia.org/wiki/AEX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    '^AMX': {'url': 'https://en.wikipedia.org/wiki/AMX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    'PSI20.LS': {'url': 'https://en.wikipedia.org/wiki/PSI-20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.LS'},
    '^IBEX': {'url': 'https://en.wikipedia.org/wiki/IBEX_35', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^OMX': {'url': 'https://en.wikipedia.org/wiki/OMX_Stockholm_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.ST'},
    '^SSMI': {'url': 'https://en.wikipedia.org/wiki/Swiss_Market_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^FTSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.L'},
    '^FTMC': {'url': 'https://en.wikipedia.org/wiki/FTSE_250_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    'XU100.IS': {'url': 'https://en.wikipedia.org/wiki/BIST_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.IS'},
    '^CASE30': {'url': 'https://en.wikipedia.org/wiki/EGX_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Reuters Code'},

    # --- Global / Raw List Types ---
    '^DJGT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50', 'type': 'table', 'company_col': 'Corporation', 'ticker_col': 'Ticker'},
    '^SPG100': {'url': 'https://en.wikipedia.org/wiki/S%26P_Global_100', 'type': 'raw_list', 'company_col': 'Company Name'},
    '^GDOW': {'url': 'https://en.wikipedia.org/wiki/The_Global_Dow', 'type': 'raw_list', 'company_col': 'Company Name'},

    # --- Other/Special ---
    '^HUI': {'url': 'https://en.wikipedia.org/wiki/HUI_Gold_Index', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol'},
    '^XAU': {'url': 'https://en.wikipedia.org/wiki/Philadelphia_Gold_and_Silver_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Trading Symbol'},
    
    # --- Skipped / Proxied ---
    '^SPG1200': {'type': 'skip', 'reason': 'No reliable list page'},
    '^VIX': {'type': 'skip', 'reason': 'Volatility measure, no companies'},
    '^IXIC': {'type': 'proxy', 'use': '^NDX', 'reason': 'No list, but 80% overlap with NASDAQ-100'}
}

# --- Main Functions ---

def scrape_table(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    for table in tables:
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(-1)
            
            if config['company_col'] in df.columns and config['ticker_col'] in df.columns:
                const_df = df[[config['company_col'], config['ticker_col']]]
                const_df.columns = ['Company Name', 'Raw Ticker']
                
                # Apply cleaning functions
                suffix = config.get('suffix', '')
                if 'clean_fn' in config:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].apply(config['clean_fn'])
                else:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
                
                return const_df[['Company Name', 'Company Ticker']]
        except:
            continue
    return pd.DataFrame()


def scrape_raw_list(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    content_div = soup.find('div', {'id': 'mw-content-text'})
    # This is a heuristic: find all list items in the main content.
    items = content_div.find_all('li')
    names = [item.get_text(strip=True).split('(')[0].strip() for item in items if len(item.get_text(strip=True)) > 2]
    # For raw lists, we'll try to find tickers in a later step
    return pd.DataFrame(names, columns=['Company Name'])


def get_ishares_csv(name):
    url = "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        content = response.content.decode('utf-8')
        first_data_line = content.find("Ticker")
        df = pd.read_csv(io.StringIO(content[first_data_line:]))
        df.dropna(subset=['Ticker'], inplace=True)
        df.rename(columns={'Name': 'Company Name', 'Ticker': 'Company Ticker'}, inplace=True)
        return df[['Company Name', 'Company Ticker']]
    except Exception as e:
        logging.error(f"Failed to scrape {name} from iShares: {e}")
        return pd.DataFrame()


def main():
    os.makedirs(OUTPUT_DIR_INDIVIDUAL, exist_ok=True)
    
    # Add a check for the input file
    if not os.path.exists(INPUT_CSV):
        logging.error(f"Input file not found: {INPUT_CSV}")
        logging.error("Please make sure 'master_indices_list.csv' exists in the correct 'data' directory.")
        return

    master_list = pd.read_csv(INPUT_CSV)
    summary_report = []
    all_constituents_dfs = []

    logging.info("--- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---")
    logging.info(f"Using Data Directory: {os.path.abspath(DATA_DIR)}")
    logging.info(f"Reading master list from: {os.path.abspath(INPUT_CSV)}")
    logging.info(f"Saving individual lists to: {os.path.abspath(OUTPUT_DIR_INDIVIDUAL)}")

    for _, row in master_list.iterrows():
        index_name, index_ticker = row['Index Name'], row['Ticker']
        
        print("\n" + "="*70)
        logging.info(f"Processing Index: {index_name} ({index_ticker})")

        safe_ticker_name = re.sub(r'[\^.:]', '_', index_ticker)
        filepath = os.path.join(OUTPUT_DIR_INDIVIDUAL, f"{safe_ticker_name}.csv")

        if os.path.exists(filepath):
            logging.info(f"  ✅ Constituent file already exists. Skipping.")
            continue

        config = INDEX_CONFIG.get(index_ticker)
        status, scraped_count = "Failed", 0
        df_constituents = pd.DataFrame()

        if not config:
            status = "No Config"
        elif config['type'] == 'skip':
            status = f"Skipped ({config['reason']})"
        elif config['type'] == 'proxy':
            status = f"Proxy ({config['use']})" # We will handle this in the next script
        elif config['type'] == 'ishares_csv':
            df_constituents = get_ishares_csv(config['name'])
            status = "Success (iShares)"
        elif config['type'] == 'table':
            df_constituents = scrape_table(config)
            status = "Success (Table)"
        elif config['type'] == 'raw_list':
            df_constituents = scrape_raw_list(config)
            status = "Success (Raw List)"

        scraped_count = len(df_constituents)
        if scraped_count > 0:
            df_constituents.to_csv(filepath, index=False)
            logging.info(f"  ✅ Saved {scraped_count} constituents to '{filepath}'")
            
        summary_report.append({"Index Name": index_name, "Status": status, "Scraped Count": scraped_count, "Source URL": config.get('url', 'N/A') if config else 'N/A'})

    # (Final report generation remains the same)
    logging.info("\n\n" + "="*80)
    logging.info("--- CONFIG-DRIVEN SCRAPING SUMMARY ---")
    report_df = pd.DataFrame(summary_report)
    print(report_df.to_string())
    report_df.to_csv(SUMMARY_REPORT_FILE, index=False)
    logging.info(f"\n✅ Full summary report saved to '{os.path.abspath(SUMMARY_REPORT_FILE)}'")

if __name__ == "__main__":
    main()

2025-11-04 15:43:19,918 - WARNING - Could not use __file__, falling back to current working directory: /home/harshvardhan/dcv3/Stock-Market-Indices/yfinancedata
2025-11-04 15:43:19,921 - INFO - --- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---
2025-11-04 15:43:19,921 - INFO - Using Data Directory: /home/harshvardhan/dcv3/data
2025-11-04 15:43:19,921 - INFO - Reading master list from: /home/harshvardhan/dcv3/data/master_indices_list.csv
2025-11-04 15:43:19,922 - INFO - Saving individual lists to: /home/harshvardhan/dcv3/data/constituent_lists
2025-11-04 15:43:19,922 - INFO - Processing Index: Dow Jones Global Titans 50 Inde (^DJGT)
2025-11-04 15:43:19,923 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:19,923 - INFO - Processing Index: S&P GLOBAL 100 ( C ) (^SPG100)
2025-11-04 15:43:19,923 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:19,924 - INFO - Processing Index: S&P GLOBAL 1200 (^SPG1200)
2025-11-04 15:43:19,925 - INF

2025-11-04 15:43:20,271 - INFO - Processing Index: Wilshire 5000 Total Market Inde (^W5000)
2025-11-04 15:43:20,272 - INFO - Processing Index: SSE Composite Index (China) (000001.SS)


2025-11-04 15:43:20,560 - INFO - Processing Index: SZSE Component Index (China) (399001.SZ)
2025-11-04 15:43:20,560 - INFO - Processing Index: CSI 300 Index (000300.SS)
2025-11-04 15:43:20,561 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:20,561 - INFO - Processing Index: SSE 50 Index (000016.SS)
2025-11-04 15:43:20,561 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:20,562 - INFO - Processing Index: HANG SENG INDEX (^HSI)
2025-11-04 15:43:20,562 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:20,562 - INFO - Processing Index: S&P BSE SENSEX (^BSESN)
2025-11-04 15:43:20,562 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:20,562 - INFO - Processing Index: NIFTY 50 (^NSEI)
2025-11-04 15:43:20,562 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:20,563 - INFO - Processing Index: NIFTY NEXT 50 (^NSMIDCP)


2025-11-04 15:43:20,868 - INFO -   ✅ Saved 50 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_NSMIDCP.csv'
2025-11-04 15:43:20,869 - INFO - Processing Index: IDX COMPOSITE (^JKSE)
2025-11-04 15:43:20,869 - INFO - Processing Index: Nikkei 225 (^N225)


2025-11-04 15:43:21,103 - INFO - Processing Index: FTSE Bursa Malaysia KLCI (^KLSE)
2025-11-04 15:43:21,103 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:21,103 - INFO - Processing Index: Tadawul All Shares Index (^TASI.SR)
2025-11-04 15:43:21,104 - INFO - Processing Index: STI Index (^STI)
2025-11-04 15:43:21,104 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:21,104 - INFO - Processing Index: KOSPI Composite Index (^KS11)


...

2025-11-04 15:43:23,776 - INFO - Processing Index: OMX Copenhagen 25 Index (^OMXC25)
2025-11-04 15:43:23,777 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:23,777 - INFO - Processing Index: OMX Helsinki 25 (^OMXH25)


2025-11-04 15:43:24,009 - INFO - Processing Index: CAC 40 (^FCHI)
2025-11-04 15:43:24,010 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,010 - INFO - Processing Index: CAC Next 20 (^CN20)
2025-11-04 15:43:24,010 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,010 - INFO - Processing Index: SBF 120 (^SBF120)
2025-11-04 15:43:24,011 - INFO - Processing Index: DAX P (^GDAXI)
2025-11-04 15:43:24,011 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,011 - INFO - Processing Index: MDAX                          P (^MDAXI)
2025-11-04 15:43:24,011 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,012 - INFO - Processing Index: TecDAX                        P (^TECDAX)


2025-11-04 15:43:24,219 - INFO - Processing Index: ISEQ All Share (^ISEQ)
2025-11-04 15:43:24,219 - INFO - Processing Index: FTSE MIB (Italy) (FTSEMIB.MI)
2025-11-04 15:43:24,219 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,220 - INFO - Processing Index: AEX (Netherlands) (^AEX)
2025-11-04 15:43:24,220 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,220 - INFO - Processing Index: AMX (Netherlands) (^AMX)
2025-11-04 15:43:24,220 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,220 - INFO - Processing Index: PSI-20 (Portugal) (PSI20.LS)
2025-11-04 15:43:24,220 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,221 - INFO - Processing Index: IBEX 35... (^IBEX)
2025-11-04 15:43:24,221 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:43:24,221 - INFO - Processing Index: OMX Stockholm 30 (Sweden) (^OMX)
2025-11-04 15:43:24,221 - INFO -   ✅ Constituent file alrea

2025-11-04 15:43:24,620 - INFO - 

2025-11-04 15:43:24,621 - INFO - --- CONFIG-DRIVEN SCRAPING SUMMARY ---
2025-11-04 15:43:24,623 - INFO - 
✅ Full summary report saved to '/home/harshvardhan/dcv3/data/constituent_scraping_summary.csv'


                                  Index Name                                      Status  Scraped Count                                                                  Source URL
0                            S&P GLOBAL 1200             Skipped (No reliable list page)              0                                                                         N/A
1                                     MERVAL                                      Failed              0                                        https://en.wikipedia.org/wiki/MERVAL
2                                   S&P IPSA                             Success (Table)             28  https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones
3                CBOE Volatility Index (VIX)  Skipped (Volatility measure, no companies)              0                                                                         N/A
4                           NASDAQ Composite                                Proxy (^NDX)            

In [5]:
import pandas as pd
import yfinance as yf
import time
import requests
import io
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Proper Path Setup ---
# Use the __file__ variable to get the directory of the currently running script
# This makes the paths portable and not hardcoded to your machine.
try:
    # Get the directory where this script is located (e.g., .../yfinancedata)
    THIS_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive environments (like Jupyter or an IDE run button)
    THIS_SCRIPT_DIR = os.getcwd()
    logging.warning(f"Could not use __file__, falling back to current working directory: {THIS_SCRIPT_DIR}")

# Get the parent directory of this script (e.g., .../Stock-Market-Indices)
PROJECT_ROOT = os.path.normpath(os.path.join(THIS_SCRIPT_DIR, '..'))

# The 'data' folder is *inside* the project root
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# --- File Definitions ---
INPUT_CSV = os.path.join(DATA_DIR, "master_indices_list.csv")
OUTPUT_DIR_INDIVIDUAL = os.path.join(DATA_DIR, "constituent_lists")
OUTPUT_FILE_MASTER = os.path.join(DATA_DIR, "master_constituents_list.csv")
SUMMARY_REPORT_FILE = os.path.join(DATA_DIR, "constituent_scraping_summary.csv")
# --- End of Path Setup ---


# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (Based on your research) ====================
# =====================================================================================
# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (v2 - EXPANDED) ==============================
# =====================================================================================
INDEX_CONFIG = {
    # --- Americas ---
    '^DJI': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^DJT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Transportation_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^DJU': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Utility_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^NDX': {'url': 'https://en.wikipedia.org/wiki/Nasdaq-100', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^RUI': {'url': 'https://en.wikipedia.org/wiki/Russell_1000_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^GSPC': {'url': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 'type': 'table', 'company_col': 'Security', 'ticker_col': 'Symbol', 'clean_fn': lambda s: s.replace('.', '-')},
    '^OEX': {'url': 'https://en.wikipedia.org/wiki/S%26P_100', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^MID': {'url': 'https://en.wikipedia.org/wiki/S%26P_400', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'},
    '^BVSP': {'url': 'https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.SA'},
    '^GSPTSE': {'url': 'https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^IPSA': {'url': 'https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones', 'type': 'table', 'company_col': 'Empresa', 'ticker_col': 'Símbolo', 'suffix': '.SN'},
    '^MXX': {'url': 'https://en.wikipedia.org/wiki/Indice_de_Precios_y_Cotizaciones', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^RUT': {'type': 'ishares_csv', 'name': 'Russell 2000'},
    '^MERV': {'url': 'https://en.wikipedia.org/wiki/MERVAL', 'type': 'navbox_list', 'company_col': 'Company Name', 'suffix': '.BA'},
    
    # --- Asia-Pacific ---
    '000300.SS': {'url': 'https://en.wikipedia.org/wiki/CSI_300_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '000016.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_50_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker symbol'},
    '000001.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'},
    '^HSI': {'url': 'https://en.wikipedia.org/wiki/Hang_Seng_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker', 'clean_fn': lambda s: f"{int(s.split(':')[-1].strip()):04d}.HK"},
    '^BSESN': {'url': 'https://en.wikipedia.org/wiki/BSE_SENSEX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.BO'},
    '^NSEI': {'url': 'https://en.wikipedia.org/wiki/NIFTY_50', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^NSMIDCP': {'url': 'https://en.wikipedia.org/wiki/NIFTY_Next_50', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^N225': {'url': 'https://en.wikipedia.org/wiki/Nikkei_225', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Symbol', 'suffix': '.T'},
    '^KLSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_Bursa_Malaysia_KLCI', 'type': 'table', 'company_col': 'Constituent Name', 'ticker_col': 'Stock Code', 'suffix': '.KL'},
    '^STI': {'url': 'https://en.wikipedia.org/wiki/Straits_Times_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Stock symbol', 'suffix': '.SI'},
    '^SET.BK': {'url': 'https://en.wikipedia.org/wiki/SET50_Index_and_SET100_Index', 'type': 'table', 'company_col': 'Securities Name', 'ticker_col': 'Symbol', 'suffix': '.BK'},
    '^AXJO': {'url': 'https://en.wikipedia.org/wiki/S%26P/ASX_200', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    '^NZ50': {'url': 'https://en.wikipedia.org/wiki/S%26P/NZX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.NZ'},
    '^KS11': {'url': 'https://en.wikipedia.org/wiki/KOSPI', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^TWII': {'url': 'https://en.wikipedia.org/wiki/Taiwan_Capitalization_Weighted_Stock_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.TW'},
    '^AORD': {'url': 'https://en.wikipedia.org/wiki/All_Ordinaries', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    
    # --- Europe ---
    '^STOXX50E': {'url': 'https://en.wikipedia.org/wiki/EURO_STOXX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^STOXX': {'url': 'https://en.wikipedia.org/wiki/STOXX_Europe_600', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^ATX': {'url': 'https://en.wikipedia.org/wiki/Austrian_Traded_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.VI'},
    '^BFX': {'url': 'https://en.wikipedia.org/wiki/BEL_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.BR'},
    '^OMXC25': {'url': 'https://en.wikipedia.org/wiki/OMX_Copenhagen_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.CO'},
    '^OMXH25': {'url': 'https://en.wikipedia.org/wiki/OMX_Helsinki_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.HE'},
    '^FCHI': {'url': 'https://en.wikipedia.org/wiki/CAC_40', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.PA'},
    '^CN20': {'url': 'https://en.wikipedia.org/wiki/CAC_Next_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.PA'},
    '^GDAXI': {'url': 'https://en.wikipedia.org/wiki/DAX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.DE'},
    '^MDAXI': {'url': 'https://en.wikipedia.org/wiki/MDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    '^TECDAX': {'url': 'https://en.wikipedia.org/wiki/TecDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    'FTSEMIB.MI': {'url': 'https://en.wikipedia.org/wiki/FTSE_MIB', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^AEX': {'url': 'https://en.wikipedia.org/wiki/AEX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    '^AMX': {'url': 'https://en.wikipedia.org/wiki/AMX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    'PSI20.LS': {'url': 'https://en.wikipedia.org/wiki/PSI-20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.LS'},
    '^IBEX': {'url': 'https://en.wikipedia.org/wiki/IBEX_35', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^OMX': {'url': 'https://en.wikipedia.org/wiki/OMX_Stockholm_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.ST'},
    '^SSMI': {'url': 'https://en.wikipedia.org/wiki/Swiss_Market_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^FTSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.L'},
    '^FTMC': {'url': 'https://en.wikipedia.org/wiki/FTSE_250_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    'XU100.IS': {'url': 'https://en.wikipedia.org/wiki/BIST_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.IS'},
    '^CASE30': {'url': 'https://en.wikipedia.org/wiki/EGX_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Reuters Code'},

    # --- Global / Raw List Types ---
    '^DJGT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50', 'type': 'table', 'company_col': 'Corporation', 'ticker_col': 'Ticker'},
    '^SPG100': {'url': 'https://en.wikipedia.org/wiki/S%26P_Global_100', 'type': 'raw_list', 'company_col': 'Company Name'},
    '^GDOW': {'url': 'https://en.wikipedia.org/wiki/The_Global_Dow', 'type': 'raw_list', 'company_col': 'Company Name'},

    # --- Other/Special ---
    '^HUI': {'url': 'https://en.wikipedia.org/wiki/HUI_Gold_Index', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol'},
    '^XAU': {'url': 'https://en.wikipedia.org/wiki/Philadelphia_Gold_and_Silver_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Trading Symbol'},
    
    # --- Skipped / Proxied ---
    '^SPG1200': {'type': 'skip', 'reason': 'No reliable list page'},
    '^VIX': {'type': 'skip', 'reason': 'Volatility measure, no companies'},
    '^IXIC': {'type': 'proxy', 'use': '^NDX', 'reason': 'No list, but 80% overlap with NASDAQ-100'}
}

# --- Main Functions ---

def scrape_table(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    for table in tables:
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(-1)
            
            if config['company_col'] in df.columns and config['ticker_col'] in df.columns:
                const_df = df[[config['company_col'], config['ticker_col']]]
                const_df.columns = ['Company Name', 'Raw Ticker']
                
                # Apply cleaning functions
                suffix = config.get('suffix', '')
                if 'clean_fn' in config:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].apply(config['clean_fn'])
                else:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
                
                return const_df[['Company Name', 'Company Ticker']]
        except:
            continue
    return pd.DataFrame()


def scrape_raw_list(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    content_div = soup.find('div', {'id': 'mw-content-text'})
    # This is a heuristic: find all list items in the main content.
    items = content_div.find_all('li')
    names = [item.get_text(strip=True).split('(')[0].strip() for item in items if len(item.get_text(strip=True)) > 2]
    # For raw lists, we'll try to find tickers in a later step
    return pd.DataFrame(names, columns=['Company Name'])


def get_ishares_csv(name):
    url = "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        content = response.content.decode('utf-8')
        first_data_line = content.find("Ticker")
        df = pd.read_csv(io.StringIO(content[first_data_line:]))
        df.dropna(subset=['Ticker'], inplace=True)
        df.rename(columns={'Name': 'Company Name', 'Ticker': 'Company Ticker'}, inplace=True)
        return df[['Company Name', 'Company Ticker']]
    except Exception as e:
        logging.error(f"Failed to scrape {name} from iShares: {e}")
        return pd.DataFrame()


def main():
    os.makedirs(OUTPUT_DIR_INDIVIDUAL, exist_ok=True)
    
    # Add a check for the input file
    if not os.path.exists(INPUT_CSV):
        logging.error(f"Input file not found: {INPUT_CSV}")
        logging.error("Please make sure 'master_indices_list.csv' exists in the correct 'data' directory.")
        return

    master_list = pd.read_csv(INPUT_CSV)
    summary_report = []
    all_constituents_dfs = []

    logging.info("--- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---")
    logging.info(f"Using Project Root: {os.path.abspath(PROJECT_ROOT)}")
    logging.info(f"Using Data Directory: {os.path.abspath(DATA_DIR)}")
    logging.info(f"Reading master list from: {os.path.abspath(INPUT_CSV)}")
    logging.info(f"Saving individual lists to: {os.path.abspath(OUTPUT_DIR_INDIVIDUAL)}")

    for _, row in master_list.iterrows():
        index_name, index_ticker = row['Index Name'], row['Ticker']
        
        print("\n" + "="*70)
        logging.info(f"Processing Index: {index_name} ({index_ticker})")

        safe_ticker_name = re.sub(r'[\^.:]', '_', index_ticker)
        filepath = os.path.join(OUTPUT_DIR_INDIVIDUAL, f"{safe_ticker_name}.csv")

        if os.path.exists(filepath):
            logging.info(f"  ✅ Constituent file already exists. Skipping.")
            continue

        config = INDEX_CONFIG.get(index_ticker)
        status, scraped_count = "Failed", 0
        df_constituents = pd.DataFrame()

        if not config:
            status = "No Config"
        elif config['type'] == 'skip':
            status = f"Skipped ({config['reason']})"
        elif config['type'] == 'proxy':
            status = f"Proxy ({config['use']})" # We will handle this in the next script
        elif config['type'] == 'ishares_csv':
            df_constituents = get_ishares_csv(config['name'])
            status = "Success (iShares)"
        elif config['type'] == 'table':
            df_constituents = scrape_table(config)
            status = "Success (Table)"
        elif config['type'] == 'raw_list':
            df_constituents = scrape_raw_list(config)
            status = "Success (Raw List)"

        scraped_count = len(df_constituents)
        if scraped_count > 0:
            df_constituents.to_csv(filepath, index=False)
            logging.info(f"  ✅ Saved {scraped_count} constituents to '{filepath}'")
            
        summary_report.append({"Index Name": index_name, "Status": status, "Scraped Count": scraped_count, "Source URL": config.get('url', 'N/A') if config else 'N/A'})

    # (Final report generation remains the same)
    logging.info("\n\n" + "="*80)
    logging.info("--- CONFIG-DRIVEN SCRAPING SUMMARY ---")
    report_df = pd.DataFrame(summary_report)
    print(report_df.to_string())
    report_df.to_csv(SUMMARY_REPORT_FILE, index=False)
    logging.info(f"\n✅ Full summary report saved to '{os.path.abspath(SUMMARY_REPORT_FILE)}'")

if __name__ == "__main__":
    main()

2025-11-04 14:50:54,792 - WARNING - Could not use __file__, falling back to current working directory: /home/harshvardhan/dcv3/Stock-Market-Indices/yfinancedata
2025-11-04 14:50:54,795 - ERROR - Input file not found: /home/harshvardhan/dcv3/Stock-Market-Indices/data/master_indices_list.csv
2025-11-04 14:50:54,795 - ERROR - Please make sure 'master_indices_list.csv' exists in the correct 'data' directory.


In [ ]:
import pandas as pd
import yfinance as yf
import time
import requests
import io
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Proper Path Setup ---
# Use the __file__ variable to get the directory of the currently running script
# This makes the paths portable and not hardcoded to your machine.
try:
    # Get the directory where this script is located (e.g., .../yfinancedata)
    THIS_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive environments (like Jupyter)
    THIS_SCRIPT_DIR = os.getcwd()
    logging.warning(f"Could not use __file__, falling back to current working directory: {THIS_SCRIPT_DIR}")


# Your original code implied SCRIPT_DIR was one level *above* this script's location
# (e.g., .../Stock-Market-Indices)
SCRIPT_DIR = os.path.normpath(os.path.join(THIS_SCRIPT_DIR, '..'))

# Your original code implied DATA_DIR was a *sibling* of SCRIPT_DIR
# (e.g., .../dcv3/data)
DATA_DIR = os.path.normpath(os.path.join(SCRIPT_DIR, '..', 'data'))

# --- File Definitions ---
INPUT_CSV = os.path.join(DATA_DIR, "master_indices_list.csv")
OUTPUT_DIR_INDIVIDUAL = os.path.join(DATA_DIR, "constituent_lists")
OUTPUT_FILE_MASTER = os.path.join(DATA_DIR, "master_constituents_list.csv")
SUMMARY_REPORT_FILE = os.path.join(DATA_DIR, "constituent_scraping_summary.csv")
# --- End of Path Setup ---


# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (Based on your research) ====================
# =====================================================================================
# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (v2 - EXPANDED) ==============================
# =====================================================================================
INDEX_CONFIG = {
    # --- Americas ---
    '^DJI': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^DJT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Transportation_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^DJU': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Utility_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^NDX': {'url': 'https://en.wikipedia.org/wiki/Nasdaq-100', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^RUI': {'url': 'https://en.wikipedia.org/wiki/Russell_1000_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^GSPC': {'url': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 'type': 'table', 'company_col': 'Security', 'ticker_col': 'Symbol', 'clean_fn': lambda s: s.replace('.', '-')},
    '^OEX': {'url': 'https://en.wikipedia.org/wiki/S%26P_100', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^MID': {'url': 'https://en.wikipedia.org/wiki/S%26P_400', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'},
    '^BVSP': {'url': 'https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.SA'},
    '^GSPTSE': {'url': 'https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    # UPDATED based on your input: 'Company' -> 'Empresa', 'Symbol' -> 'Símbolo'
    '^IPSA': {'url': 'https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.SN'},
    '^MXX': {'url': 'https://en.wikipedia.org/wiki/Indice_de_Precios_y_Cotizaciones', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^RUT': {'type': 'ishares_csv', 'name': 'Russell 2000'},
    '^MERV': {'url': 'https://en.wikipedia.org/wiki/MERVAL', 'type': 'navbox_list', 'company_col': 'Company Name', 'suffix': '.BA'},
    
    # --- Asia-Pacific ---
    '000300.SS': {'url': 'https://en.wikipedia.org/wiki/CSI_300_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '000016.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_50_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker symbol'},
    '000001.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker Symbol'}, #No company list
    '^HSI': {'url': 'https://en.wikipedia.org/wiki/Hang_Seng_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker', 'clean_fn': lambda s: f"{int(s.split(':')[-1].strip()):04d}.HK"},
    '^BSESN': {'url': 'https://en.wikipedia.org/wiki/BSE_SENSEX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.BO'},
    '^NSEI': {'url': 'https://en.wikipedia.org/wiki/NIFTY_50', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    # UPDATED based on your input: 'Company Name' -> 'Company name'
    '^NSMIDCP': {'url': 'https://en.wikipedia.org/wiki/NIFTY_Next_50', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    # UPDATED based on your input: 'Symbol' -> 'Code'
    '^N225': {'url': 'https://en.wikipedia.org/wiki/Nikkei_225', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Code', 'suffix': '.T'},
    '^KLSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_Bursa_Malaysia_KLCI', 'type': 'table', 'company_col': 'Constituent Name', 'ticker_col': 'Stock Code', 'suffix': '.KL'},
    '^STI': {'url': 'https://en.wikipedia.org/wiki/Straits_Times_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Stock symbol', 'suffix': '.SI'},
    '^SET.BK': {'url': 'https://en.wikipedia.org/wiki/SET50_Index_and_SET100_Index', 'type': 'table', 'company_col': 'Securities Name', 'ticker_col': 'Symbol', 'suffix': '.BK'},
    '^AXJO': {'url': 'https://en.wikipedia.org/wiki/S%26P/ASX_200', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    '^NZ50': {'url': 'https://en.wikipedia.org/wiki/S%26P/NZX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.NZ'},
    '^KS11': {'url': 'https://en.wikipedia.org/wiki/KOSPI', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^TWII': {'url': 'https://en.wikipedia.org/wiki/Taiwan_Capitalization_Weighted_Stock_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.TW'},
    '^AORD': {'url': 'https://en.wikipedia.org/wiki/All_Ordinaries', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    
    # --- Europe ---
    '^STOXX50E': {'url': 'https://en.wikipedia.org/wiki/EURO_STOXX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^STOXX': {'url': 'https://en.wikipedia.org/wiki/STOXX_Europe_600', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^ATX': {'url': 'https://en.wikipedia.org/wiki/Austrian_Traded_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.VI'},
    # UPDATED based on your input: 'Ticker' -> 'Ticker symbol'
    '^BFX': {'url': 'https://en.wikipedia.org/wiki/BEL_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.BR'},
    '^OMXC25': {'url': 'https://en.wikipedia.org/wiki/OMX_Copenhagen_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.CO'},
    # UPDATED based on your input: 'Ticker' -> 'Ticker symbol'
    '^OMXH25': {'url': 'https://en.wikipedia.org/wiki/OMX_Helsinki_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.HE'},
    # UPDATED based on your input: 'Ticker' -> 'Ticker symbol'
    '^FCHI': {'url': 'https://en.wikipedia.org/wiki/CAC_40', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.PA'},
    '^CN20': {'url': 'https://en.wikipedia.org/wiki/CAC_Next_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.PA'},
    '^GDAXI': {'url': 'https://en.wikipedia.org/wiki/DAX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.DE'},
    '^MDAXI': {'url': 'https://en.wikipedia.org/wiki/MDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    '^TECDAX': {'url': 'https://en.wikipedia.org/wiki/TecDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    'FTSEMIB.MI': {'url': 'https://en.wikipedia.org/wiki/FTSE_MIB', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^AEX': {'url': 'https://en.wikipedia.org/wiki/AEX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    '^AMX': {'url': 'https://en.wikipedia.org/wiki/AMX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    'PSI20.LS': {'url': 'https://en.wikipedia.org/wiki/PSI-20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.LS'},
    # UPDATED based on your input: 'Company' -> 'Empresa'
    '^IBEX': {'url': 'https://en.wikipedia.org/wiki/IBEX_35', 'type': 'table', 'company_col': 'Empresa', 'ticker_col': 'Ticker'},
    '^OMX': {'url': 'https://en.wikipedia.org/wiki/OMX_Stockholm_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.ST'},
    '^SSMI': {'url': 'https://en.wikipedia.org/wiki/Swiss_Market_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^FTSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.L'},
    '^FTMC': {'url': 'https://en.wikipedia.org/wiki/FTSE_250_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    'XU100.IS': {'url': 'https://en.wikipedia.org/wiki/BIST_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.IS'},
    '^CASE30': {'url': 'https://en.wikipedia.org/wiki/EGX_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Reuters Code'},

    # --- Global / Raw List Types ---
    '^DJGT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50', 'type': 'table', 'company_col': 'Corporation', 'ticker_col': 'Ticker'},
    '^SPG100': {'url': 'https://en.wikipedia.org/wiki/S%26P_Global_100', 'type': 'raw_list', 'company_col': 'Company Name'},
    '^GDOW': {'url': 'https://en.wikipedia.org/wiki/The_Global_Dow', 'type': 'raw_list', 'company_col': 'Company Name'},

    # --- Other/Special ---
    '^HUI': {'url': 'https://en.wikipedia.org/wiki/HUI_Gold_Index', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol'},
    '^XAU': {'url': 'https://en.wikipedia.org/wiki/Philadelphia_Gold_and_Silver_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Trading Symbol'},
    
    # --- Skipped / Proxied ---
    '^SPG1200': {'type': 'skip', 'reason': 'No reliable list page'},
    '^VIX': {'type': 'skip', 'reason': 'Volatility measure, no companies'},
    '^IXIC': {'type': 'proxy', 'use': '^NDX', 'reason': 'No list, but 80% overlap with NASDAQ-100'}
}

# --- Main Functions ---

def scrape_table(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    for table in tables:
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(-1)
            
            if config['company_col'] in df.columns and config['ticker_col'] in df.columns:
                const_df = df[[config['company_col'], config['ticker_col']]]
                const_df.columns = ['Company Name', 'Raw Ticker']
                
                # Apply cleaning functions
                suffix = config.get('suffix', '')
                if 'clean_fn' in config:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].apply(config['clean_fn'])
                else:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
                
                return const_df[['Company Name', 'Company Ticker']]
        except:
            continue
    return pd.DataFrame()


def scrape_raw_list(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    content_div = soup.find('div', {'id': 'mw-content-text'})
    # This is a heuristic: find all list items in the main content.
    items = content_div.find_all('li')
    names = [item.get_text(strip=True).split('(')[0].strip() for item in items if len(item.get_text(strip=True)) > 2]
    # For raw lists, we'll try to find tickers in a later step
    return pd.DataFrame(names, columns=['Company Name'])


def get_ishares_csv(name):
    url = "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        content = response.content.decode('utf-8')
        first_data_line = content.find("Ticker")
        df = pd.read_csv(io.StringIO(content[first_data_line:]))
        df.dropna(subset=['Ticker'], inplace=True)
        df.rename(columns={'Name': 'Company Name', 'Ticker': 'Company Ticker'}, inplace=True)
        return df[['Company Name', 'Company Ticker']]
    except Exception as e:
        logging.error(f"Failed to scrape {name} from iShares: {e}")
        return pd.DataFrame()


def main():
    os.makedirs(OUTPUT_DIR_INDIVIDUAL, exist_ok=True)
    
    # Add a check for the input file
    if not os.path.exists(INPUT_CSV):
        logging.error(f"Input file not found: {INPUT_CSV}")
        logging.error("Please make sure 'master_indices_list.csv' exists in the correct 'data' directory.")
        return

    master_list = pd.read_csv(INPUT_CSV)
    summary_report = []
    all_constituents_dfs = []

    logging.info("--- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---")
    logging.info(f"Using Data Directory: {os.path.abspath(DATA_DIR)}")
    logging.info(f"Reading master list from: {os.path.abspath(INPUT_CSV)}")
    logging.info(f"Saving individual lists to: {os.path.abspath(OUTPUT_DIR_INDIVIDUAL)}")

    for _, row in master_list.iterrows():
        index_name, index_ticker = row['Index Name'], row['Ticker']
        
        print("\n" + "="*70)
        logging.info(f"Processing Index: {index_name} ({index_ticker})")

        safe_ticker_name = re.sub(r'[\^.:]', '_', index_ticker)
        filepath = os.path.join(OUTPUT_DIR_INDIVIDUAL, f"{safe_ticker_name}.csv")

        if os.path.exists(filepath):
            logging.info(f"  ✅ Constituent file already exists. Skipping.")
            continue

        config = INDEX_CONFIG.get(index_ticker)
        status, scraped_count = "Failed", 0
        df_constituents = pd.DataFrame()

        if not config:
            status = "No Config"
        elif config['type'] == 'skip':
            status = f"Skipped ({config['reason']})"
        elif config['type'] == 'proxy':
            status = f"Proxy ({config['use']})" # We will handle this in the next script
        elif config['type'] == 'ishares_csv':
            df_constituents = get_ishares_csv(config['name'])
            status = "Success (iShares)"
        elif config['type'] == 'table':
            df_constituents = scrape_table(config)
            status = "Success (Table)"
        elif config['type'] == 'raw_list':
            df_constituents = scrape_raw_list(config)
            status = "Success (Raw List)"

        scraped_count = len(df_constituents)
        if scraped_count > 0:
            df_constituents.to_csv(filepath, index=False)
            logging.info(f"  ✅ Saved {scraped_count} constituents to '{filepath}'")
            
        summary_report.append({"Index Name": index_name, "Status": status, "Scraped Count": scraped_count, "Source URL": config.get('url', 'N/A') if config else 'N/A'})

    # (Final report generation remains the same)
    logging.info("\n\n" + "="*80)
    logging.info("--- CONFIG-DRIVEN SCRAPING SUMMARY ---")
    report_df = pd.DataFrame(summary_report)
    print(report_df.to_string())
    report_df.to_csv(SUMMARY_REPORT_FILE, index=False)
    logging.info(f"\n✅ Full summary report saved to '{os.path.abspath(SUMMARY_REPORT_FILE)}'")

if __name__ == "__main__":
    main()

2025-11-04 15:48:03,697 - WARNING - Could not use __file__, falling back to current working directory: /home/harshvardhan/dcv3/Stock-Market-Indices/yfinancedata
2025-11-04 15:48:03,701 - INFO - --- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---
2025-11-04 15:48:03,701 - INFO - Using Data Directory: /home/harshvardhan/dcv3/data
2025-11-04 15:48:03,702 - INFO - Reading master list from: /home/harshvardhan/dcv3/data/master_indices_list.csv
2025-11-04 15:48:03,702 - INFO - Saving individual lists to: /home/harshvardhan/dcv3/data/constituent_lists
2025-11-04 15:48:03,702 - INFO - Processing Index: Dow Jones Global Titans 50 Inde (^DJGT)
/tmp/ipykernel_8404/958607470.py:152: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  const_df['Company Tic

2025-11-04 15:48:04,032 - INFO -   ✅ Saved 109 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_SPG100.csv'
2025-11-04 15:48:04,032 - INFO - Processing Index: S&P GLOBAL 1200 (^SPG1200)
2025-11-04 15:48:04,033 - INFO - Processing Index: The Global Dow (USD) (^GDOW)


2025-11-04 15:48:04,242 - INFO -   ✅ Saved 151 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_GDOW.csv'
2025-11-04 15:48:04,243 - INFO - Processing Index: MERVAL (^MERV)
2025-11-04 15:48:04,244 - INFO - Processing Index: IBOVESPA (^BVSP)
/tmp/ipykernel_8404/958607470.py:152: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
2025-11-04 15:48:04,407 - INFO -   ✅ Saved 88 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_BVSP.csv'
2025-11-04 15:48:04,407 - INFO - Processing Index: S&P/TSX Composite index (^GSPTSE)


/tmp/ipykernel_8404/958607470.py:152: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
2025-11-04 15:48:04,673 - INFO -   ✅ Saved 223 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_GSPTSE.csv'
2025-11-04 15:48:04,674 - INFO - Processing Index: S&P IPSA (^IPSA)


2025-11-04 15:48:04,879 - INFO - Processing Index: IPC MEXICO (^MXX)
/tmp/ipykernel_8404/958607470.py:152: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
2025-11-04 15:48:05,042 - INFO -   ✅ Saved 34 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_MXX.csv'
2025-11-04 15:48:05,043 - INFO - Processing Index: CBOE Volatility Index (VIX) (^VIX)
2025-11-04 15:48:05,043 - INFO - Processing Index: Dow Jones Industrial Average (^DJI)


...

2025-11-04 15:48:25,408 - INFO -   ✅ Saved 100 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_FTSE.csv'
2025-11-04 15:48:25,409 - INFO - Processing Index: FTSE 250 (^FTMC)


2025-11-04 15:48:25,783 - INFO -   ✅ Saved 250 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_FTMC.csv'
2025-11-04 15:48:25,783 - INFO - Processing Index: UK FTSE All Share (^FTAS)
2025-11-04 15:48:25,784 - INFO - Processing Index: Amex Oil Index (Energy) (^XOI)
2025-11-04 15:48:25,784 - INFO - Processing Index: PHLX Semiconductor (^SOX)
2025-11-04 15:48:25,785 - INFO - Processing Index: HUI Gold Index (Metals) (^HUI)


2025-11-04 15:48:25,995 - INFO -   ✅ Saved 18 constituents to '/home/harshvardhan/dcv3/data/constituent_lists/_HUI.csv'
2025-11-04 15:48:25,995 - INFO - Processing Index: PHLX Gold/Silver Sector (^XAU)


2025-11-04 15:48:26,203 - INFO - 

2025-11-04 15:48:26,203 - INFO - --- CONFIG-DRIVEN SCRAPING SUMMARY ---
2025-11-04 15:48:26,205 - INFO - 
✅ Full summary report saved to '/home/harshvardhan/dcv3/data/constituent_scraping_summary.csv'


                                  Index Name                                      Status  Scraped Count                                                                  Source URL
0            Dow Jones Global Titans 50 Inde                             Success (Table)             50                    https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50
1                       S&P GLOBAL 100 ( C )                          Success (Raw List)            109                              https://en.wikipedia.org/wiki/S%26P_Global_100
2                            S&P GLOBAL 1200             Skipped (No reliable list page)              0                                                                         N/A
3                       The Global Dow (USD)                          Success (Raw List)            151                                https://en.wikipedia.org/wiki/The_Global_Dow
4                                     MERVAL                                      Failed            

In [ ]:
import pandas as pd
import yfinance as yf
import time
import requests
import io
import os
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Proper Path Setup ---
# Use the __file__ variable to get the directory of the currently running script
# This makes the paths portable and not hardcoded to your machine.
try:
    # Get the directory where this script is located (e.g., .../yfinancedata)
    THIS_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for interactive environments (like Jupyter)
    THIS_SCRIPT_DIR = os.getcwd()
    logging.warning(f"Could not use __file__, falling back to current working directory: {THIS_SCRIPT_DIR}")


# Your original code implied SCRIPT_DIR was one level *above* this script's location
# (e.g., .../Stock-Market-Indices)
SCRIPT_DIR = os.path.normpath(os.path.join(THIS_SCRIPT_DIR, '..'))

# Your original code implied DATA_DIR was a *sibling* of SCRIPT_DIR
# (e.g., .../dcv3/data)
DATA_DIR = os.path.normpath(os.path.join(SCRIPT_DIR, '..', 'data'))

# --- File Definitions ---
INPUT_CSV = os.path.join(DATA_DIR, "master_indices_list.csv")
OUTPUT_DIR_INDIVIDUAL = os.path.join(DATA_DIR, "constituent_lists")
OUTPUT_FILE_MASTER = os.path.join(DATA_DIR, "master_constituents_list.csv")
SUMMARY_REPORT_FILE = os.path.join(DATA_DIR, "constituent_scraping_summary.csv")
# --- End of Path Setup ---


# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (Based on your research) ====================
# =====================================================================================
# =====================================================================================
# === THE GROUND-TRUTH CONFIGURATION HUB (v2 - EXPANDED) ==============================
# =====================================================================================
#
# UPDATED: This list has been filtered to ONLY include the 43 indices
# from your provided text list. All others have been removed.
#
INDEX_CONFIG = {
    # --- Americas ---
    '^DJI': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^DJT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Transportation_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^DJU': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Utility_Average', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^NDX': {'url': 'https://en.wikipedia.org/wiki/Nasdaq-100', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^RUI': {'url': 'https://en.wikipedia.org/wiki/Russell_1000_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol'},
    '^OEX': {'url': 'https://en.wikipedia.org/wiki/S%26P_100', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol'},
    '^BVSP': {'url': 'https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.SA'},
    '^GSPTSE': {'url': 'https://en.wikipedia.org/wiki/S%26P/TSX_Composite_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^IPSA': {'url': 'https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.SN'},
    
    # --- Asia-Pacific ---
    '000300.SS': {'url': 'https://en.wikipedia.org/wiki/CSI_300_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '000016.SS': {'url': 'https://en.wikipedia.org/wiki/SSE_50_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker symbol'},
    '^HSI': {'url': 'https://en.wikipedia.org/wiki/Hang_Seng_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker', 'clean_fn': lambda s: f"{int(s.split(':')[-1].strip()):04d}.HK"},
    '^BSESN': {'url': 'https://en.wikipedia.org/wiki/BSE_SENSEX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.BO'},
    '^NSEI': {'url': 'https://en.wikipedia.org/wiki/NIFTY_50', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^NSMIDCP': {'url': 'https://en.wikipedia.org/wiki/NIFTY_Next_50', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Symbol', 'suffix': '.NS'},
    '^N225': {'url': 'https://en.wikipedia.org/wiki/Nikkei_225', 'type': 'table', 'company_col': 'Company Name', 'ticker_col': 'Code', 'suffix': '.T'},
    '^KLSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_Bursa_Malaysia_KLCI', 'type': 'table', 'company_col': 'Constituent Name', 'ticker_col': 'Stock Code', 'suffix': '.KL'},
    '^STI': {'url': 'https://en.wikipedia.org/wiki/Straits_Times_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Stock symbol', 'suffix': '.SI'},
    '^SET.BK': {'url': 'https://en.wikipedia.org/wiki/SET50_Index_and_SET100_Index', 'type': 'table', 'company_col': 'Securities Name', 'ticker_col': 'Symbol', 'suffix': '.BK'},
    '^AXJO': {'url': 'https://en.wikipedia.org/wiki/S%26P/ASX_200', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Code', 'suffix': '.AX'},
    '^NZ50': {'url': 'https://en.wikipedia.org/wiki/S%26P/NZX_50', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.NZ'},
    
    # --- Europe ---
    '^STOXX50E': {'url': 'https://en.wikipedia.org/wiki/EURO_STOXX_50', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Ticker'},
    '^ATX': {'url': 'https://en.wikipedia.org/wiki/Austrian_Traded_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.VI'},
    '^BFX': {'url': 'https://en.wikipedia.org/wiki/BEL_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.BR'},
    '^OMXC25': {'url': 'https://en.wikipedia.org/wiki/OMX_Copenhagen_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.CO'},
    '^OMXH25': {'url': 'https://en.wikipedia.org/wiki/OMX_Helsinki_25', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.HE'},
    '^FCHI': {'url': 'https://en.wikipedia.org/wiki/CAC_40', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.PA'},
    '^CN20': {'url': 'https://en.wikipedia.org/wiki/CAC_Next_20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.PA'},
    '^GDAXI': {'url': 'https://en.wikipedia.org/wiki/DAX', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.DE'},
    '^MDAXI': {'url': 'https://en.wikipedia.org/wiki/MDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    '^TECDAX': {'url': 'https://en.wikipedia.org/wiki/TecDAX', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Symbol', 'suffix': '.DE'},
    'FTSEMIB.MI': {'url': 'https://en.wikipedia.org/wiki/FTSE_MIB', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^AEX': {'url': 'https://en.wikipedia.org/wiki/AEX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    '^AMX': {'url': 'https://en.wikipedia.org/wiki/AMX_index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker symbol', 'suffix': '.AS'},
    'PSI20.LS': {'url': 'https://en.wikipedia.org/wiki/PSI-20', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.LS'},
    '^IBEX': {'url': 'https://en.wikipedia.org/wiki/IBEX_35', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker'},
    '^OMX': {'url': 'https://en.wikipedia.org/wiki/OMX_Stockholm_30', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Symbol', 'suffix': '.ST'},
    '^FTSE': {'url': 'https://en.wikipedia.org/wiki/FTSE_100_Index', 'type': 'table', 'company_col': 'Company', 'ticker_col': 'Ticker', 'suffix': '.L'},

    # --- Global / Raw List Types ---
    '^DJGT': {'url': 'https://en.wikipedia.org/wiki/Dow_Jones_Global_Titans_50', 'type': 'table', 'company_col': 'Corporation', 'ticker_col': 'Ticker'},
    '^SPG100': {'url': 'https://en.wikipedia.org/wiki/S%26P_Global_100', 'type': 'raw_list', 'company_col': 'Company Name'},
    '^GDOW': {'url': 'https://en.wikipedia.org/wiki/The_Global_Dow', 'type': 'raw_list', 'company_col': 'Company Name'},

    # --- Other/Special ---
    '^HUI': {'url': 'https://en.wikipedia.org/wiki/HUI_Gold_Index', 'type': 'table', 'company_col': 'Company name', 'ticker_col': 'Symbol'},
    '^XAU': {'url': 'https://en.wikipedia.org/wiki/Philadelphia_Gold_and_Silver_Index', 'type': 'table', 'company_col': 'Name', 'ticker_col': 'Trading Symbol'},
}


# --- Main Functions ---

def scrape_table(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    for table in tables:
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(-1)
            
            if config['company_col'] in df.columns and config['ticker_col'] in df.columns:
                const_df = df[[config['company_col'], config['ticker_col']]]
                const_df.columns = ['Company Name', 'Raw Ticker']
                
                # Apply cleaning functions
                suffix = config.get('suffix', '')
                if 'clean_fn' in config:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].apply(config['clean_fn'])
                else:
                    const_df['Company Ticker'] = const_df['Raw Ticker'].astype(str) + suffix
                
                return const_df[['Company Name', 'Company Ticker']]
        except:
            continue
    return pd.DataFrame()


def scrape_raw_list(config):
    response = requests.get(config['url'], headers={'User-Agent': 'MyCoolTool/1.0'})
    soup = BeautifulSoup(response.text, 'lxml')
    content_div = soup.find('div', {'id': 'mw-content-text'})
    # This is a heuristic: find all list items in the main content.
    items = content_div.find_all('li')
    names = [item.get_text(strip=True).split('(')[0].strip() for item in items if len(item.get_text(strip=True)) > 2]
    # For raw lists, we'll try to find tickers in a later step
    return pd.DataFrame(names, columns=['Company Name'])


def get_ishares_csv(name):
    url = "httpsNext_50://www.ishares.com/us/products/239710/ishares-russell-2000-etf/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        content = response.content.decode('utf-8')
        first_data_line = content.find("Ticker")
        df = pd.read_csv(io.StringIO(content[first_data_line:]))
        df.dropna(subset=['Ticker'], inplace=True)
        df.rename(columns={'Name': 'Company Name', 'Ticker': 'Company Ticker'}, inplace=True)
        return df[['Company Name', 'Company Ticker']]
    except Exception as e:
        logging.error(f"Failed to scrape {name} from iShares: {e}")
        return pd.DataFrame()


def main():
    os.makedirs(OUTPUT_DIR_INDIVIDUAL, exist_ok=True)
    
    # Add a check for the input file
    if not os.path.exists(INPUT_CSV):
        logging.error(f"Input file not found: {INPUT_CSV}")
        logging.error("Please make sure 'master_indices_list.csv' exists in the correct 'data' directory.")
        return

    master_list = pd.read_csv(INPUT_CSV)
    summary_report = []
    all_constituents_dfs = []

    logging.info("--- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---")
    logging.info(f"Using Data Directory: {os.path.abspath(DATA_DIR)}")
    logging.info(f"Reading master list from: {os.path.abspath(INPUT_CSV)}")
    logging.info(f"Saving individual lists to: {os.path.abspath(OUTPUT_DIR_INDIVIDUAL)}")

    for _, row in master_list.iterrows():
        index_name, index_ticker = row['Index Name'], row['Ticker']
        
        print("\n" + "="*70)
        logging.info(f"Processing Index: {index_name} ({index_ticker})")

        safe_ticker_name = re.sub(r'[\^.:]', '_', index_ticker)
        filepath = os.path.join(OUTPUT_DIR_INDIVIDUAL, f"{safe_ticker_name}.csv")

        if os.path.exists(filepath):
            logging.info(f"  ✅ Constituent file already exists. Skipping.")
            continue

        config = INDEX_CONFIG.get(index_ticker)
        status, scraped_count = "Failed", 0
        df_constituents = pd.DataFrame()

        if not config:
            status = "No Config"
        elif config.get('type') == 'skip': # Use .get() for safety
            status = f"Skipped ({config.get('reason', 'N/A')})"
        elif config.get('type') == 'proxy':
            status = f"Proxy ({config.get('use', 'N/A')})" # We will handle this in the next script
        elif config.get('type') == 'ishares_csv':
            df_constituents = get_ishares_csv(config['name'])
            status = "Success (iShares)"
        elif config.get('type') == 'table':
            df_constituents = scrape_table(config)
            status = "Success (Table)"
        elif config.get('type') == 'raw_list':
            df_constituents = scrape_raw_list(config)
            status = "Success (Raw List)"
        else:
            # This handles cases where the ticker is in master_list but not in our filtered INDEX_CONFIG
            status = "No Config (Filtered)"


        scraped_count = len(df_constituents)
        if scraped_count > 0:
            df_constituents.to_csv(filepath, index=False)
            logging.info(f"  ✅ Saved {scraped_count} constituents to '{filepath}'")
            
        summary_report.append({"Index Name": index_name, "Status": status, "Scraped Count": scraped_count, "Source URL": config.get('url', 'N/A') if config else 'N/A'})

    # (Final report generation remains the same)
    logging.info("\n\n" + "="*80)
    logging.info("--- CONFIG-DRIVEN SCRAPING SUMMARY ---")
    report_df = pd.DataFrame(summary_report)
    print(report_df.to_string())
    report_df.to_csv(SUMMARY_REPORT_FILE, index=False)
    logging.info(f"\n✅ Full summary report saved to '{os.path.abspath(SUMMARY_REPORT_FILE)}'")

if __name__ == "__main__":
    main()

2025-11-04 15:54:21,826 - WARNING - Could not use __file__, falling back to current working directory: /home/harshvardhan/dcv3/Stock-Market-Indices/yfinancedata
2025-11-04 15:54:21,828 - INFO - --- Starting Phase 2a: CONFIG-DRIVEN Constituent Scraping ---
2025-11-04 15:54:21,828 - INFO - Using Data Directory: /home/harshvardhan/dcv3/data
2025-11-04 15:54:21,829 - INFO - Reading master list from: /home/harshvardhan/dcv3/data/master_indices_list.csv
2025-11-04 15:54:21,829 - INFO - Saving individual lists to: /home/harshvardhan/dcv3/data/constituent_lists
2025-11-04 15:54:21,829 - INFO - Processing Index: Dow Jones Global Titans 50 Inde (^DJGT)
2025-11-04 15:54:21,830 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:21,830 - INFO - Processing Index: S&P GLOBAL 100 ( C ) (^SPG100)
2025-11-04 15:54:21,830 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:21,830 - INFO - Processing Index: S&P GLOBAL 1200 (^SPG1200)
2025-11-04 15:54:21,831 - INF

2025-11-04 15:54:22,186 - INFO - Processing Index: IDX COMPOSITE (^JKSE)
2025-11-04 15:54:22,187 - INFO - Processing Index: Nikkei 225 (^N225)


2025-11-04 15:54:22,482 - INFO - Processing Index: FTSE Bursa Malaysia KLCI (^KLSE)
2025-11-04 15:54:22,482 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:22,483 - INFO - Processing Index: Tadawul All Shares Index (^TASI.SR)
2025-11-04 15:54:22,483 - INFO - Processing Index: STI Index (^STI)
2025-11-04 15:54:22,483 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:22,484 - INFO - Processing Index: KOSPI Composite Index (^KS11)
2025-11-04 15:54:22,484 - INFO - Processing Index: TWSE Capitalization Weighted Stock Index (^TWII)
2025-11-04 15:54:22,484 - INFO - Processing Index: SET_SET Index (^SET.BK)
2025-11-04 15:54:22,484 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:22,485 - INFO - Processing Index: BIST 100 (Turkey) (XU100.IS)
2025-11-04 15:54:22,485 - INFO - Processing Index: ALL ORDINARIES (^AORD)
2025-11-04 15:54:22,485 - INFO - Processing Index: S&P/ASX 200 (^AXJO)
2025-11-04 15:54:22,485 - INFO -   ✅ Con

2025-11-04 15:54:22,799 - INFO - Processing Index: STXE 600                      I (^STOXX)
2025-11-04 15:54:22,799 - INFO - Processing Index: EGX 30 Price Return Index (^CASE30)
2025-11-04 15:54:22,800 - INFO - Processing Index: Austrian Traded Index in EUR (^ATX)
2025-11-04 15:54:22,965 - INFO - Processing Index: BEL 20 (^BFX)
2025-11-04 15:54:22,965 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:22,966 - INFO - Processing Index: OMX Copenhagen 25 Index (^OMXC25)
2025-11-04 15:54:22,966 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:22,966 - INFO - Processing Index: OMX Helsinki 25 (^OMXH25)


2025-11-04 15:54:23,196 - INFO - Processing Index: CAC 40 (^FCHI)
2025-11-04 15:54:23,392 - INFO - Processing Index: CAC Next 20 (^CN20)
2025-11-04 15:54:23,392 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,393 - INFO - Processing Index: SBF 120 (^SBF120)
2025-11-04 15:54:23,393 - INFO - Processing Index: DAX P (^GDAXI)
2025-11-04 15:54:23,393 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,393 - INFO - Processing Index: MDAX                          P (^MDAXI)
2025-11-04 15:54:23,393 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,394 - INFO - Processing Index: TecDAX                        P (^TECDAX)


2025-11-04 15:54:23,545 - INFO - Processing Index: ISEQ All Share (^ISEQ)
2025-11-04 15:54:23,546 - INFO - Processing Index: FTSE MIB (Italy) (FTSEMIB.MI)
2025-11-04 15:54:23,546 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,546 - INFO - Processing Index: AEX (Netherlands) (^AEX)
2025-11-04 15:54:23,547 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,547 - INFO - Processing Index: AMX (Netherlands) (^AMX)
2025-11-04 15:54:23,547 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,547 - INFO - Processing Index: PSI-20 (Portugal) (PSI20.LS)
2025-11-04 15:54:23,547 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,548 - INFO - Processing Index: IBEX 35... (^IBEX)


2025-11-04 15:54:23,799 - INFO - Processing Index: OMX Stockholm 30 (Sweden) (^OMX)
2025-11-04 15:54:23,799 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,800 - INFO - Processing Index: SMI PR (^SSMI)
2025-11-04 15:54:23,800 - INFO - Processing Index: FTSE 100 (^FTSE)
2025-11-04 15:54:23,800 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,800 - INFO - Processing Index: FTSE 250 (^FTMC)
2025-11-04 15:54:23,801 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,801 - INFO - Processing Index: UK FTSE All Share (^FTAS)
2025-11-04 15:54:23,801 - INFO - Processing Index: Amex Oil Index (Energy) (^XOI)
2025-11-04 15:54:23,801 - INFO - Processing Index: PHLX Semiconductor (^SOX)
2025-11-04 15:54:23,802 - INFO - Processing Index: HUI Gold Index (Metals) (^HUI)
2025-11-04 15:54:23,802 - INFO -   ✅ Constituent file already exists. Skipping.
2025-11-04 15:54:23,802 - INFO - Processing Index: PHLX Gold/Silver Sector (










                                  Index Name           Status  Scraped Count                                                                  Source URL
0                            S&P GLOBAL 1200        No Config              0                                                                         N/A
1                                     MERVAL        No Config              0                                                                         N/A
2                                   S&P IPSA  Success (Table)              0  https://en.wikipedia.org/wiki/%C3%8Dndice_de_Precios_Selectivo_de_Acciones
3                CBOE Volatility Index (VIX)        No Config              0                                                                         N/A
4                           NASDAQ Composite        No Config              0                                                                         N/A
5                               Russell 3000        No Config            